In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

# add parent folder (production) to sys.path
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa, MultiBAMv4

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\my_lora_utils.py


In [ ]:
# LoRa 파라미터 설정 (generate.ipynb와 동일하게)
sf = 9
bw = 250_000  # 250 kHz
OSF = 4
fs = int(bw * OSF)  # 1 MHz

print(f"SF = {sf}, BW = {bw/1e3:.1f} kHz, fs = {fs/1e6:.3f} MHz")

# LoRa 인스턴스 생성 (on-the-fly 노이즈 추가용)
lora = LoRa(sf, bw)

In [ ]:
########################################
##  1. 클린 IQ 데이터 로드            ##
########################################

base_dir = "dataset_v4_sf9_bw250k"
iq_dir = os.path.join(base_dir, "clean_iq")

print("Loading clean IQ data...")
iq_files = sorted(glob.glob(os.path.join(iq_dir, "*.npy")))
print(f"Found {len(iq_files)} clean IQ files")

# 클린 IQ 로드
iq_data_list = []
for f in iq_files:
    x_clean = np.load(f)  # complex IQ
    iq_data_list.append(x_clean)

X_clean_iq = np.array(iq_data_list)  # shape: (512, 2048) or similar
print(f"Clean IQ shape: {X_clean_iq.shape}, dtype: {X_clean_iq.dtype}")


In [ ]:
########################################
##  2. 클린 스펙트로그램 생성         ##
########################################

print("\nGenerating clean spectrograms from IQ...")

X_clean_spec_list = []
for x_iq in X_clean_iq:
    # IQ → 스펙트로그램 변환 (my_lora_utils의 함수 사용)
    spec = generate_spectrogram(x_iq, fs)  # shape: (256, 17) 등
    X_clean_spec_list.append(spec)

X_clean_spec = np.array(X_clean_spec_list)
print(f"Clean spectrogram shape: {X_clean_spec.shape}")

# Flatten to 1D for training
X_clean_flat = np.array([s.flatten() for s in X_clean_spec])
print(f"Clean flat shape: {X_clean_flat.shape}")  # (512, 4352)

In [ ]:
########################################
##  3. On-the-fly 노이즈 추가        ##
########################################

print("\nAdding noise on-the-fly...")

# SNR 범위 설정
SNR_MIN = -30
SNR_MAX = 15

# 각 클린 IQ에 랜덤 SNR로 노이즈 추가
X_noisy_iq_list = []
snr_list = []

for x_clean_iq in X_clean_iq:
    # 랜덤 SNR 선택
    snr = np.random.randint(SNR_MIN, SNR_MAX + 1)
    snr_list.append(snr)
    
    # 노이즈 추가
    x_noisy_iq = lora.awgn_iq(x_clean_iq, snr)
    X_noisy_iq_list.append(x_noisy_iq)

X_noisy_iq = np.array(X_noisy_iq_list)
print(f"Noisy IQ shape: {X_noisy_iq.shape}")
print(f"SNR range used: {min(snr_list)} ~ {max(snr_list)} dB")


In [ ]:
########################################
##  4. 노이즈 스펙트로그램 생성       ##
########################################

print("\nGenerating noisy spectrograms...")

X_noisy_spec_list = []
for x_noisy_iq in X_noisy_iq:
    spec = generate_spectrogram(x_noisy_iq, fs)
    X_noisy_spec_list.append(spec)

X_noisy_spec = np.array(X_noisy_spec_list)
print(f"Noisy spectrogram shape: {X_noisy_spec.shape}")

# Flatten to 1D
X_noisy_flat = np.array([s.flatten() for s in X_noisy_spec])
print(f"Noisy flat shape: {X_noisy_flat.shape}")  # (512, 4352)

In [ ]:
########################################
##  5. BAMv4 디노이징 학습           ##
########################################

# 모델 하이퍼파라미터
input_dim = X_clean_flat.shape[1]  # 4352 (256 * 17)
hidden_dims = [2048, 512]  # 4352 → 2048 → 512 (compressed)

print(f"\n=== Training BAMv4 Denoising Model ===")
print(f"Input dim: {input_dim}")
print(f"Hidden dims: {hidden_dims}")
print(f"Architecture: {input_dim} → {' → '.join(map(str, hidden_dims))}")

# BAMv4 모델 생성
model = MultiBAMv4(
    input_dim=input_dim,
    hidden_dims=hidden_dims,
    activation="tanh",
    device=None  # auto-detect cuda/cpu
)

print(f"Device: {model.device}")

# 학습 플래그
TRAIN = True  # 학습 실행 여부

if TRAIN:
    print("\n=== Starting Training ===")
    
    # 학습 파라미터
    num_epochs = 10
    batch_size = 64
    lr = 1e-3
    
    # Denoising 학습
    history = model.fit_denoise(
        X_noisy=X_noisy_flat,
        X_clean=X_clean_flat,
        num_epochs=num_epochs,
        batch_size=batch_size,
        lr=lr,
        weight_decay=0.0,
        verbose=True
    )
    
    print("\n=== Training Complete ===")
    
    # Loss 플롯
    plt.figure(figsize=(10, 4))
    plt.plot(history)
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('BAMv4 Denoising Training Loss')
    plt.grid(True)
    plt.show()
else:
    print("\nTraining skipped (TRAIN=False)")


In [ ]:
########################################
##  6. 모델 저장                     ##
########################################

import torch

if TRAIN:
    # 모델 저장 폴더
    weight_folder = "weights_bamv4"
    os.makedirs(weight_folder, exist_ok=True)
    
    # PyTorch 모델 전체 저장
    model_path = os.path.join(weight_folder, "multibam_v4_denoise.pth")
    torch.save(model.state_dict(), model_path)
    print(f"\n✅ Model saved to: {model_path}")
    
    # 아키텍처 정보도 함께 저장
    config = {
        'input_dim': input_dim,
        'hidden_dims': hidden_dims,
        'activation': 'tanh',
        'sf': sf,
        'bw': bw,
        'fs': fs
    }
    config_path = os.path.join(weight_folder, "model_config.npy")
    np.save(config_path, config)
    print(f"✅ Config saved to: {config_path}")
    
print("\n=== How to Load Model ===")
print("""
# 모델 로드 예시:
config = np.load('weights_bamv4/model_config.npy', allow_pickle=True).item()
model = MultiBAMv4(
    input_dim=config['input_dim'],
    hidden_dims=config['hidden_dims'],
    activation=config['activation']
)
model.load_state_dict(torch.load('weights_bamv4/multibam_v4_denoise.pth'))
model.eval()
""")


In [ ]:
########################################
##  7. 테스트: 디노이징 성능 확인    ##
########################################

# 테스트 샘플 선택
test_idx = 0

print(f"\n=== Testing Denoising on Sample {test_idx} ===")
print(f"SNR used: {snr_list[test_idx]} dB")

# 입력/타겟 준비
x_noisy = X_noisy_flat[test_idx:test_idx+1]  # (1, 4352)
x_clean = X_clean_flat[test_idx:test_idx+1]  # (1, 4352)

# 모델 예측
x_denoised = model.reconstruct_numpy(x_noisy)  # (1, 4352)

# Reshape back to spectrogram
spec_noisy = x_noisy.reshape(256, 17)
spec_clean = x_clean.reshape(256, 17)
spec_denoised = x_denoised.reshape(256, 17)

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im1 = axes[0].imshow(spec_noisy, aspect='auto', cmap='viridis')
axes[0].set_title(f'Noisy (SNR={snr_list[test_idx]} dB)')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Frequency')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(spec_denoised, aspect='auto', cmap='viridis')
axes[1].set_title('Denoised (BAMv4 Output)')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Frequency')
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(spec_clean, aspect='auto', cmap='viridis')
axes[2].set_title('Clean (Target)')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Frequency')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

# MSE 계산
mse_noisy = np.mean((spec_noisy - spec_clean) ** 2)
mse_denoised = np.mean((spec_denoised - spec_clean) ** 2)

print(f"\nMSE (Noisy vs Clean): {mse_noisy:.6f}")
print(f"MSE (Denoised vs Clean): {mse_denoised:.6f}")
print(f"Improvement: {(1 - mse_denoised/mse_noisy)*100:.2f}%")
